# TTS Experiment — Facebook MMS TTS

**WeatherSpeak PH** — Gemma 4 Hackathon

## Objective

Evaluate Facebook's Massively Multilingual Speech (MMS) TTS models as a replacement
for Coqui XTTS v2. MMS provides **native** Cebuano and Tagalog models — no Spanish
phoneme approximation required.

### Models used
| Language | Model | Params |
|---|---|---|
| Cebuano | `facebook/mms-tts-ceb` | 36.3M |
| Tagalog | `facebook/mms-tts-tgl` | 36.3M |
| English | `facebook/mms-tts-eng` | 36.3M |

### Bulletin
`PAGASA_20-19W_Pepito_SWB#01` — Severe Weather Bulletin, Tropical Depression Pepito

### Synthesis order
CEB → TL → EN

### License note
MMS TTS models are CC-BY-NC 4.0 (non-commercial). Acceptable for this hackathon.
Flag for production licensing review.

In [ ]:
import re
import time
import numpy as np
import torch
from pathlib import Path
from pydub import AudioSegment
from transformers import VitsModel, AutoTokenizer

# --- Paths ---
notebook_dir = Path(".")
output_dir = notebook_dir / "08-mms-tts-experiment"
output_dir.mkdir(exist_ok=True)

data_dir = notebook_dir.parent / "data"
input_dir = data_dir / "radio_bulletins"

# --- Experiment scope ---
STEM = "PAGASA_20-19W_Pepito_SWB#01"
LANGUAGES = ["ceb", "tl", "en"]  # synthesis order: CEB first
MMS_MODELS = {
    "ceb": "facebook/mms-tts-ceb",
    "tl":  "facebook/mms-tts-tgl",
    "en":  "facebook/mms-tts-eng",
}

# --- Verify input files exist ---
input_files = {lang: input_dir / f"{STEM}_radio_{lang}.md" for lang in LANGUAGES}
missing = [str(p) for p in input_files.values() if not p.exists()]
if missing:
    print(f"⚠  Missing input files: {missing}")
else:
    print(f"✓ All 3 input files found")
    for lang, p in input_files.items():
        print(f"  {lang}: {p.name}")

print(f"✓ Output dir: {output_dir.absolute()}")

## 1. Text Preprocessing

Reuse `preprocess_for_tts` from notebook 07 — strips markdown formatting so
nothing is read aloud except actual spoken content.

In [ ]:
def preprocess_for_tts(markdown_text: str) -> str:
    """Strip markdown formatting for clean TTS input.

    Nothing in the output should be read aloud unless it is actual spoken content.
    Modal-ready: pure function, no external state.
    Copied from notebook 07 — keep in sync if updated there.
    """
    text = markdown_text

    # Remove stage directions: **(Sound effect: ...)** on their own line
    text = re.sub(r"^\s*\*\*\([^)]+\)\*\*\s*$", "", text, flags=re.MULTILINE)

    # Remove role labels: **BROADCASTER:** **Boses:** **LABEL:** etc.
    text = re.sub(r"\*\*[A-Za-z][A-Za-z\s]+:\*\*\s*", "", text)

    # Section headings → just the heading text as a plain spoken sentence
    text = re.sub(r"^#{1,6}\s+(.+)$", r"\1.", text, flags=re.MULTILINE)

    # Horizontal rules — must come BEFORE bold/italic removal to avoid *** corruption
    text = re.sub(r"^[-*_]{3,}$", "", text, flags=re.MULTILINE)

    # Bold and italic markers
    text = re.sub(r"\*{1,3}(.+?)\*{1,3}", r"\1", text)
    text = re.sub(r"_{1,3}(.+?)_{1,3}", r"\1", text)

    # Inline code
    text = re.sub(r"`(.+?)`", r"\1", text)

    # Blockquotes
    text = re.sub(r"^>\s+", "", text, flags=re.MULTILINE)

    # Numbered and bulleted list markers
    text = re.sub(r"^\s*\d+\.\s+", "", text, flags=re.MULTILINE)
    text = re.sub(r"^\s*[-*+]\s+", "", text, flags=re.MULTILINE)

    # Remove any leftover parenthetical stage directions (fallback, after bold stripped)
    text = re.sub(r"^\s*\([^)]{10,}\)\s*$", "", text, flags=re.MULTILINE)

    # Remove role labels that survived bold stripping: WORD: at start of line
    text = re.sub(r"^[A-Z][A-Za-z\s]+:\s*$", "", text, flags=re.MULTILINE)

    # Collapse 3+ blank lines to 2
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


# Preprocess all 3 scripts upfront and save plain text files
plain_texts = {}
for lang in LANGUAGES:
    md = input_files[lang].read_text(encoding="utf-8")
    plain = preprocess_for_tts(md)
    plain_texts[lang] = plain

    plain_path = output_dir / f"{STEM}_radio_{lang}_plain.txt"
    plain_path.write_text(plain, encoding="utf-8")
    print(f"✓ {lang}: {len(md)} chars → {len(plain)} chars  (saved {plain_path.name})")

# Preview CEB plain text
print("\nPREPROCESSED TEXT — CEB (first 500 chars)")
print("=" * 60)
print(plain_texts["ceb"][:500])
print("...")

## 2. Load MMS Models

Load all three VITS models simultaneously. Each is ~140 MB (F32). Downloads are cached
by HuggingFace after first run.

Unlike XTTS v2 (1.8 GB single multilingual model), MMS uses one small model per language.

In [ ]:
print("Loading MMS models (downloads ~140 MB each on first run, cached after)...")
t0 = time.time()

models = {}
tokenizers = {}

for lang in LANGUAGES:
    model_id = MMS_MODELS[lang]
    lang_label = {"ceb": "Cebuano", "tl": "Tagalog", "en": "English"}[lang]
    print(f"  Loading {lang_label} ({model_id})...")
    t_lang = time.time()
    tokenizers[lang] = AutoTokenizer.from_pretrained(model_id)
    models[lang] = VitsModel.from_pretrained(model_id)
    models[lang].eval()
    print(f"  ✓ {lang_label} loaded in {time.time() - t_lang:.1f}s")

print(f"\n✓ All 3 models loaded in {time.time() - t0:.1f}s total")
print(f"  Sample rates: { {lang: models[lang].config.sampling_rate for lang in LANGUAGES} }")

## 3. Synthesis

`synthesize_with_mms`: tokenize → model inference → float32 waveform → int16 PCM →
pydub AudioSegment → MP3 at 128kbps.

No chunking needed — VITS processes the full text in one pass (unlike XTTS v2's
~200-char chunk limit).

Synthesis order: CEB → TL → EN.

In [ ]:
def synthesize_with_mms(
    text: str,
    model,
    tokenizer,
    output_path: Path,
    sample_rate: int | None = None,
) -> Path:
    """Synthesize plain text to MP3 using a HuggingFace VitsModel.

    Modal-ready: all inputs are primitive types + Path; no notebook globals.

    Args:
        text: Plain text (no markdown). Use preprocess_for_tts() first.
        model: Loaded VitsModel instance.
        tokenizer: Loaded AutoTokenizer instance.
        output_path: Destination MP3 path.
        sample_rate: Override model's native sample rate if needed.

    Returns:
        output_path on success.
    """
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        waveform = model(**inputs).waveform

    rate = sample_rate or model.config.sampling_rate

    # float32 [-1, 1] → int16 PCM for pydub
    pcm = (waveform.squeeze().numpy() * 32_767).clip(-32_768, 32_767).astype(np.int16)

    segment = AudioSegment(
        pcm.tobytes(),
        frame_rate=rate,
        sample_width=2,  # 16-bit = 2 bytes
        channels=1,
    )
    segment.export(str(output_path), format="mp3", bitrate="128k")
    return output_path


# --- Run synthesis: CEB → TL → EN ---
results = {}

for lang in LANGUAGES:
    lang_label = {"ceb": "Cebuano", "tl": "Tagalog", "en": "English"}[lang]
    output_path = output_dir / f"{STEM}_radio_{lang}.mp3"

    print(f"Synthesizing {lang_label}...")
    t_start = time.time()
    synthesize_with_mms(
        plain_texts[lang],
        models[lang],
        tokenizers[lang],
        output_path,
    )
    elapsed = time.time() - t_start

    size_kb = output_path.stat().st_size // 1024
    duration_s = (size_kb * 1024 * 8) / 128_000

    print(f"  ✓ {elapsed:.1f}s  |  {size_kb} KB  |  ~{duration_s:.0f}s audio")
    print(f"  ✓ {output_path.name}")

    results[lang] = {
        "lang_label": lang_label,
        "elapsed_s": round(elapsed, 1),
        "size_kb": size_kb,
        "duration_s": round(duration_s),
        "path": output_path,
    }

print("\n✓ Synthesis complete")

## 4. Manual Audio Assessment

Listen to each MP3 using the players below, then fill in your scores in the
assessment dict in the next cell.

**Score guide:**
- `quality_score` (1–5): Overall audio quality and clarity
- `natural_filipino` (1–5): How natural the Filipino pronunciation sounds (CEB/TL only;
  use for EN to rate overall naturalness)

Run the cell after filling in your scores — the comparison table in the next section
uses these values.

In [ ]:
from IPython.display import Audio, display

for lang in LANGUAGES:
    lang_label = results[lang]["lang_label"]
    print(f"--- {lang_label} ---")
    display(Audio(str(results[lang]["path"]), autoplay=False))
    print()

In [ ]:
# ── FILL IN YOUR SCORES AFTER LISTENING ──────────────────────────────────────
assessment = {
    "ceb": {
        "quality_score": None,     # 1–5: overall audio quality
        "natural_filipino": None,  # 1–5: naturalness of Cebuano pronunciation
        "notes": "",
    },
    "tl": {
        "quality_score": None,
        "natural_filipino": None,  # 1–5: naturalness of Tagalog pronunciation
        "notes": "",
    },
    "en": {
        "quality_score": None,
        "natural_filipino": None,  # 1–5: naturalness / clarity
        "notes": "",
    },
}
# ─────────────────────────────────────────────────────────────────────────────

print("Assessment recorded. Run the next cell to see the comparison table.")
for lang, scores in assessment.items():
    label = {"ceb": "Cebuano", "tl": "Tagalog", "en": "English"}[lang]
    q = scores["quality_score"] or "—"
    n = scores["natural_filipino"] or "—"
    print(f"  {label}: quality={q}/5  naturalness={n}/5  notes={scores['notes']!r}")

## 5. Comparison: MMS TTS vs XTTS v2

XTTS v2 baseline from notebook 07 (CEB only — EN/TL were not run to completion):
- Synthesis time: 821s for CEB bulletin
- Audio duration: ~467s (~7.8 min)
- Language mapping: Spanish phoneme approximation for CEB and TL
- Model size: 1.87 GB download

MMS TTS: native Cebuano and Tagalog phonemes, ~140 MB per model.

In [ ]:
# XTTS v2 baseline from notebook 07 (CEB only)
xtts_baseline = {
    "ceb": {"elapsed_s": 821.2, "size_kb": 7293, "duration_s": 467,
            "quality_score": None, "natural_filipino": None,
            "phoneme": "es (Spanish approx)", "model_size": "1.87 GB"},
}

print("MMS TTS vs XTTS v2 — PAGASA_20-19W_Pepito_SWB#01")
print("=" * 90)
print(f"{'Model':<12} {'Lang':<10} {'Time':>7} {'Audio':>7} {'Size':>8}  "
      f"{'Quality':>8} {'Naturalness':>12}  {'Phoneme'}")
print("-" * 90)

for lang in LANGUAGES:
    lang_label = results[lang]["lang_label"]
    r = results[lang]
    a = assessment[lang]
    mms_phoneme = "native" if lang in ("ceb", "tl") else "native"
    q = f"{a['quality_score']}/5" if a["quality_score"] else "—"
    n = f"{a['natural_filipino']}/5" if a["natural_filipino"] else "—"
    print(f"{'MMS':<12} {lang_label:<10} {r['elapsed_s']:>6.1f}s {r['duration_s']:>6}s "
          f"{r['size_kb']:>7}KB  {q:>8} {n:>12}  {mms_phoneme}")

print()
for lang in ["ceb"]:
    b = xtts_baseline[lang]
    lang_label = "Cebuano"
    q = f"{b['quality_score']}/5" if b["quality_score"] else "—"
    n = f"{b['natural_filipino']}/5" if b["natural_filipino"] else "—"
    print(f"{'XTTS v2':<12} {lang_label:<10} {b['elapsed_s']:>6.1f}s {b['duration_s']:>6}s "
          f"{b['size_kb']:>7}KB  {q:>8} {n:>12}  {b['phoneme']}")

print("=" * 90)
print(f"\nMMS total model footprint: ~{len(LANGUAGES) * 140} MB  vs  XTTS v2: 1,870 MB")

# Speedup for CEB (the one language with an XTTS v2 baseline)
if "ceb" in results and results["ceb"]["elapsed_s"] > 0:
    speedup = xtts_baseline["ceb"]["elapsed_s"] / results["ceb"]["elapsed_s"]
    print(f"MMS CEB speedup vs XTTS v2: {speedup:.1f}×")

## Conclusion

> **TODO (fill in after listening):** Based on the scores above, does MMS TTS replace
> XTTS v2 for WeatherSpeak PH?
>
> - CEB/TL naturalness: is native phoneme support a meaningful improvement over Spanish approximation?
> - Speed: is the synthesis time reduction significant enough to matter for the pipeline?
> - Recommendation: replace XTTS v2 / use MMS for CEB+TL only / keep XTTS v2